# Phát hiện bất thường công nghiệp — TN1, TN2, TN2b, TN5 (Google Colab)

**Phương pháp:** Autoencoder · PaDiM · PatchCore (code trong `src/tn`) **Dữ liệu:** MVTec AD — `screw`, `grid`, `transistor`

| Thí nghiệm | Nội dung | Kết quả lưu |
|---|---|---|
| **TN1** | Tập train "không sạch": trộn 0 / 5 / 10 / 20 % ảnh lỗi vào train | `results/tn1_<method>.csv` |
| **TN2** | Few-shot: train với 1, 2, 4, 8, 16 ảnh good (+ toàn bộ để đối chiếu) | `results/tn2_<method>.csv` |
| **TN2b** | Kiểm chứng giả thuyết góc xoay của screw: đo góc từng ảnh + chạy lại few-shot trên ảnh đã căn góc | `results/tn2rot_all.csv`, `tn2rot_scores.csv` |
| **TN5** | Phân tích định tính: heatmap, ca FP/FN, AUROC theo từng loại lỗi | `results/figures/tn5/`, `tn5_per_defect_auroc.csv` |

**Cách dùng:** chọn GPU T4 rồi chạy từ trên xuống. Kết quả ghi vào Drive; nếu Colab ngắt thì chạy lại, các run đã xong tự bỏ qua. Nên chạy 2 lượt: `METHODS = ['padim', 'patchcore']` (khoảng 1 giờ), rồi `['ae']` (khoảng 2 giờ); cuối cùng chạy phần biểu đồ và TN5.

## 1. Cài đặt
Kết nối Drive, lấy code từ repo (nhánh `an/tn-refactor`) và kiểm tra GPU. Sau một lỗi CUDA phải *Disconnect and delete runtime* rồi chạy lại từ đầu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/AnDpTri/mvtec-anomaly-detection.git'
BRANCH = 'an/tn-refactor'
if subprocess.run(['git', '-C', '/content/repo', 'pull', '-q'], capture_output=True).returncode != 0:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'], check=True)
sys.path.insert(0, '/content/repo')

from src.tn.config import check_gpu
check_gpu()

## 2. Cấu hình

In [ ]:
from pathlib import Path
from src.tn.results import Results

DRIVE_ROOT = Path('/content/drive/MyDrive/MVtecAD_Deep_Learning')
DATA_ROOT  = Path('/content/data/MVTec_AD')                 # chép ra ổ local của Colab (đọc nhanh hơn Drive)
RESULTS    = DRIVE_ROOT / 'results'
MAPS_DIR   = RESULTS / 'maps'
FIG_DIR    = RESULTS / 'figures'
for p in (RESULTS, MAPS_DIR, FIG_DIR): p.mkdir(parents=True, exist_ok=True)
res = Results(RESULTS)

CATEGORIES = ['screw', 'grid', 'transistor']
METHODS    = ['padim', 'patchcore']        # lượt 1 | lượt 2: ['ae']

TN1_CATEGORIES   = ['screw', 'grid']        # transistor chỉ có 40 ảnh lỗi, không đủ để trộn vào train
TN1_RATIOS       = [0.0, 0.05, 0.10, 0.20]  # tỷ lệ ảnh lỗi TRONG tập train
TN1_SEEDS        = [0, 1, 2, 3, 4]
MIN_TEST_DEFECTS = 25                       # test phải còn ít nhất chừng này ảnh lỗi

TN2_SHOTS = [1, 2, 4, 8, 16, 'all']         # 'all' = toàn bộ train, dùng lại cho TN5
TN2_SEEDS = [0, 1, 2]

## 3. Dữ liệu
Giải nén `mvtec_3cat.zip` từ Drive (3 thư mục `screw/`, `grid/`, `transistor/` ở cấp đầu). Không có zip thì tự tải bản đầy đủ (~4,9 GB) rồi lưu zip lên Drive cho lần sau.

In [ ]:
from src.tn import data as D
D.prepare_mvtec(DATA_ROOT, CATEGORIES, DRIVE_ROOT / 'mvtec_3cat.zip')

## 4. TN1: tập train "không sạch"
Với mỗi seed, trích trước một kho ảnh lỗi chia đều theo loại lỗi, bỏ khỏi test ở **mọi** tỷ lệ. Trong cùng seed tập test giống hệt nhau và 5% ⊂ 10% ⊂ 20%, nên chênh lệch chỉ đến từ lượng ảnh lỗi trong train. Tỷ lệ làm test còn ít hơn `MIN_TEST_DEFECTS` ảnh lỗi tự bị bỏ (grid không có 20%).

In [ ]:
from src.tn.experiments import CategoryCache, run_tn1, run_tn2, prepare_rotation, run_tn2b

cache = CategoryCache(DATA_ROOT, METHODS)     # đặc trưng mỗi category chỉ trích một lần
run_tn1(res, cache, TN1_CATEGORIES, METHODS, TN1_RATIOS, TN1_SEEDS, MIN_TEST_DEFECTS)

## 5. TN2: few-shot
Mức `'all'` chỉ chạy seed 0 và lưu anomaly map cho TN5.

In [ ]:
run_tn2(res, cache, CATEGORIES, METHODS, TN2_SHOTS, TN2_SEEDS, MAPS_DIR)

## 6. TN2b: góc xoay của screw
Ở TN2, với k ≤ 8, I-AUROC của screw dưới 0,5. Giả thuyết: điểm bị góc xoay chi phối. Hai cách kiểm chứng: (1) **tương quan** giữa điểm và khoảng cách góc tới ảnh train gần nhất; (2) **can thiệp**: căn mọi ảnh về cùng tư thế (vít nằm ngang, đầu bên trái) rồi chạy lại few-shot với đúng các ảnh train của TN2. Chỉ chạy PaDiM và PatchCore (khoảng 20–30 phút).

In [ ]:
import numpy as np
from src.tn import analysis as A

R = prepare_rotation(cache)
labels_scr = np.array([t['label'] for t in R['C']['data']['test']])
print('Góc trung bình — test good:', R['ang_te'][labels_scr == 0].mean().round(),
      '| test lỗi:', R['ang_te'][labels_scr == 1].mean().round())
A.plot_alignment_check(R, FIG_DIR / 'tn2b_alignment_check.png')   # hàng dưới: mọi vít nằm ngang, đầu bên trái

In [ ]:
run_tn2b(res, R, TN2_SHOTS, TN2_SEEDS)

### Phân tích TN2b

In [ ]:
import pandas as pd
rot = pd.read_csv(res.path('tn2rot', 'all')); rot['shots'] = rot['shots'].astype(str)
sc = pd.read_csv(RESULTS / 'tn2rot_scores.csv'); sc['shots'] = sc['shots'].astype(str)
sc = sc[sc.key.isin(rot.key)]                    # bỏ dòng thừa nếu một run bị ngắt giữa chừng

display(rot.groupby(['method', 'variant', 'shots'])[['i_auroc', 'aupro']].agg(['mean', 'std']).round(3))
A.plot_rotation(rot, FIG_DIR / 'tn2b_rotation_fewshot.png')

corr = A.rotation_correlation(sc)
corr.to_csv(RESULTS / 'tn2b_correlation.csv'); display(corr)
print('spearman_rho > 0: góc càng lệch xa ảnh train thì điểm càng cao. kc_goc_tb: khoảng cách góc trung bình.')
A.plot_score_vs_angle(sc, FIG_DIR / 'tn2b_score_vs_angle.png')

## 7. Biểu đồ TN1 và TN2 (trung bình ± độ lệch chuẩn theo seed)

In [ ]:
METRICS = [('i_auroc', 'Image AUROC'), ('p_auroc', 'Pixel AUROC'), ('aupro', 'AUPRO@0.3'), ('i_f1max', 'Image F1-max')]

tn1 = res.load('tn1')
if len(tn1):
    tn1['ratio_pct'] = tn1['ratio'] * 100
    A.plot_curves(tn1, 'ratio_pct', '% ảnh lỗi trong train', sorted(tn1.category.unique()),
                  FIG_DIR / 'tn1_contamination.png', METRICS)
    display(tn1.groupby(['category', 'method', 'ratio'])[['i_auroc', 'p_auroc', 'aupro', 'i_f1max']].agg(['mean', 'std']).round(3))

tn2 = res.load('tn2')
if len(tn2):
    tn2['shots'] = tn2['shots'].astype(str)
    tn2['x'] = A.shots_to_x(tn2.shots)
    A.plot_curves(tn2, 'x', 'số ảnh train (log2)', sorted(tn2.category.unique()), FIG_DIR / 'tn2_fewshot.png',
                  METRICS, xticks=([0, 1, 2, 3, 4, 6], A.SHOT_ORDER))
    display(tn2.groupby(['category', 'method', 'shots'])[['i_auroc', 'p_auroc', 'aupro']].agg(['mean', 'std']).round(3))

## 8. TN5: phân tích định tính
Dùng anomaly map đã lưu từ TN2 (mức `'all'`, seed 0); cần chạy đủ cả 3 phương pháp trước.
1. AUROC theo từng loại lỗi. 2. Ca thất bại tại ngưỡng F1-max: FN (ảnh lỗi điểm thấp nhất), FP (ảnh good điểm cao nhất). 3. Lưới heatmap, thang màu mỗi phương pháp chung cho cả tập test.

In [ ]:
from src.tn.metrics import per_defect_auroc

TN5_DIR = FIG_DIR / 'tn5'; TN5_DIR.mkdir(exist_ok=True)
rows = []
for cat in CATEGORIES:
    data = D.load_mvtec(DATA_ROOT, cat); M = A.load_maps(MAPS_DIR, cat)
    labels = [t['label'] for t in data['test']]; dtypes = [t['dtype'] for t in data['test']]
    for m, d in M.items():
        for dt, v in per_defect_auroc(d['scores'], labels, dtypes).items():
            rows.append(dict(category=cat, method=m, defect=dt, n=dtypes.count(dt), i_auroc=v))
pdf = pd.DataFrame(rows)
if len(pdf):
    tab = pdf.pivot_table(index=['category', 'defect', 'n'], columns='method', values='i_auroc').round(3)
    tab.to_csv(RESULTS / 'tn5_per_defect_auroc.csv'); display(tab)

In [ ]:
for cat in CATEGORIES:
    M = A.load_maps(MAPS_DIR, cat)
    if not M: print(f'[{cat}] chưa có anomaly map — chạy TN2 (mức all) trước'); continue
    C = cache.get(cat, feats=False); labels = [t['label'] for t in C['data']['test']]
    for m in M:
        fn, fp, thr = A.failure_cases(M[m]['scores'], labels)
        A.draw_grid(C, M, fn + fp, f'{cat}: ca thất bại của {m.upper()} (ngưỡng F1-max = {thr:.2f})',
                    TN5_DIR / f'{cat}_fail_{m}.png')
    rng = np.random.default_rng(0); ex = []          # mỗi loại lỗi 1 ví dụ (chọn cố định)
    for dt in sorted({t['dtype'] for t in C['data']['test'] if t['label']}):
        ids = [i for i, t in enumerate(C['data']['test']) if t['dtype'] == dt]
        ex.append((int(rng.choice(ids)), ''))
    A.draw_grid(C, M, ex, f'{cat}: mỗi loại lỗi một ví dụ', TN5_DIR / f'{cat}_per_defect.png')